*ENGLISH PROFICIENCY PART*

In [1]:
#1. CREATE AN ENGLISH SKILLS CLASSIFIER

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# 1) Simulate sample data
np.random.seed(42)
n = 400
df = pd.DataFrame({
    "vocab_score_10":        np.random.randint(0, 11, size=n),
    "grammar_score_8":       np.random.randint(0, 9, size=n),
    
    "reading_inference_6":   np.random.randint(0, 7, size=n),
    
    "writing_mechanics_5":   np.random.randint(0, 6, size=n)
})

# 2) Normalize to 0–1
df["vocab_norm"]     = df["vocab_score_10"] / 10
df["grammar_norm"]   = df["grammar_score_8"] / 8
df["reading_norm"]   = df["reading_inference_6"] / 6

df["writing_norm"]   = df["writing_mechanics_5"] / 5


skill_cols = [
    "vocab_norm","grammar_norm","reading_norm",
    "writing_norm"
]

# 3) Create label based on average skill level
df["avg_skill"] = df[skill_cols].mean(axis=1)

def to_label(x):
    if x >= 0.75: return 2   # High
    elif x >= 0.50: return 1 # Medium
    else: return 0           # Low

df["proficiency_label"] = df["avg_skill"].apply(to_label)

# 4) Train classifier
X = df[skill_cols]
y = df["proficiency_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

# 5) Save model for Streamlit
import joblib
joblib.dump(clf, "english_proficiency_model.pkl")



              precision    recall  f1-score   support

           0      0.865     0.918     0.891        49
           1      0.792     0.844     0.817        45
           2      0.000     0.000     0.000         6

    accuracy                          0.830       100
   macro avg      0.552     0.588     0.569       100
weighted avg      0.780     0.830     0.804       100



/Users/jonathanlee/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/jonathanlee/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/jonathanlee/opt/anaconda3/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

['english_proficiency_model.pkl']

In [2]:
#2 RECOMMENDATION RULES

LOW, MID = 0.60, 0.80
FEEDBACK = {
    "Vocabulary": {"Needs Work": "Do 20 vocab cards/day + 5 collocations.",
                   "Adequate": "Keep 10 cards/day, mix B1–B2 sets.",
                   "Strong": "Advance to B2–C1 vocabulary and review weekly."},
    "Grammar": {"Needs Work": "15 tense/cloze per day + S–V agreement.",
                "Adequate": "8 mixed grammar exercises/day.",
                "Strong": "Complex clauses 2×/week."},
    "Sentence Structure": {"Needs Work": "2 ordering exercises/day.",
                           "Adequate": "1 ordering + clause combining.",
                           "Strong": "Paragraph cohesion tasks."},

    "Reading": {"Needs Work": "Cloze + inference on graded texts.",
                "Adequate": "Alternate cloze/inference exercises.",
                "Strong": "Longer texts + summaries."},
    "Writing": {"Needs Work": "Fix 5 sentences/day.",
                "Adequate": "Write one paragraph/day.",
                "Strong": "Weekly essay or reflection."},
}
def level_from_score(s):
    if s < LOW: return "Needs Work"
    if s < MID: return "Adequate"
    return "Strong"

def build_recommendations(scores: dict, top_k: int = 3):
    """
    scores = {'Vocabulary':0.7, 'Grammar':0.4, ...}
    returns top-k weakest skills and their recommendations
    """
    items = []
    for skill, s in scores.items():
        lvl = level_from_score(s)
        if s < LOW: gap = LOW - s
        elif s < MID: gap = MID - s
        else: gap = 1.0 - s
        items.append({
            "skill": skill,
            "score": round(s, 2),
            "level": lvl,
            "gap": round(gap, 2),
            "recommendation": FEEDBACK.get(skill, {}).get(lvl, "Practice regularly.")
        })
    items = sorted(items, key=lambda x: x["gap"], reverse=True)
    return items[:top_k], items
